# Clasificación del corpus MTSK con DeepSeek (prompt del Space original)

Replica el *prompt* y la llamada del Space `luzelorreaga/clasificador-mtsk-deepseek` sobre los 278 documentos del corpus depurado.
No necesita GPU. Guarde su clave como secreto de Colab con el nombre **DEEPSEEK_API_KEY**.

In [ ]:
!pip install -q openai openpyxl

In [ ]:
import os, re, time, json, datetime
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from google.colab import userdata

DATA_PATH   = "corpus_MTSK_depurado.xlsx"   # ← el MISMO archivo usado en la validación cruzada
OUT_CSV     = "deepseek_predicciones.csv"
MODEL       = "deepseek-chat"
TEMPERATURE = 0.2        # igual que el Space; use 0.0 para mayor determinismo
MAX_TOKENS  = 512
WORKERS     = 5

client = OpenAI(api_key=userdata.get("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")

In [ ]:
# Prompt copiado literalmente del app.py del Space
SYSTEM_PROMPT = """
Actúas como un experto en Didáctica de las Matemáticas y en el modelo MTSK (Mathematics Teacher Specialised Knowledge). Tu tarea es clasificar textos académicos (artículos, capítulos de libro, tesis, etc.) en una de las siguientes cinco categorías temáticas, definidas a partir del modelo MTSK:

**T1: Aplicaciones de MTSK en la formación de profesores**
Estudios que usan el MTSK para diseñar, implementar o evaluar programas formativos (formación inicial o continua de profesores de matemáticas de escuela, no formadores). Incluye contextos como: formación inicial, formación continuada, desarrollo profesional, talleres, cursos, etc.

**T2: Investigaciones sobre el formador de profesores de matemáticas**
Estudios cuyo foco es el conocimiento especializado, perfil, práctica o formación de quien forma a futuros profesores o a profesores en ejercicio (el formador de profesores).

**T3: MTSK en relación con distintos tópicos y etapas**
Análisis del conocimiento del profesor sobre contenidos matemáticos específicos (ej., fracciones, álgebra, geometría, simetrías, razón, área-perímetro) o en niveles educativos concretos (infantil, primaria, secundaria, universidad), sin énfasis en un programa formativo.

**T4: Desarrollo del MTSK**
Trabajos que refinan, amplían o discuten críticamente el propio modelo teórico MTSK (nuevos subdominios, categorías, indicadores, relaciones internas, discusiones teóricas, propuestas de inclusión de nuevos dominios como el afectivo).

**T5: Extensiones del MTSK**
Adaptaciones del modelo a otros contextos no estrictamente matemáticos (ej., pensamiento computacional, biología, física, educación inclusiva, interdisciplina) o articulaciones/relaciones del MTSK con otros modelos teóricos (ej., ETM, MKT, teoría de situaciones didácticas).

**Reglas de clasificación:**
1. Asigna UNA categoría principal (la que mejor represente el énfasis central del texto).
2. Opcionalmente, asigna categorías secundarias si el texto aborda múltiples temáticas.
3. En caso de duda entre dos categorías, prioriza según este orden: T4 > T5 > T2 > T1 > T3.
4. Justifica brevemente la clasificación con elementos concretos del resumen o palabras clave.

**Formato de respuesta esperado:**
- Título: [título del texto]
- Categoría principal: [T1/T2/T3/T4/T5]
- Categorías secundarias (si aplica): [T1/T2/T3/T4/T5 o "Ninguna"]
- Justificación breve: [2-3 líneas explicando por qué]
"""

In [ ]:
df = pd.read_excel(DATA_PATH).reset_index(drop=True)
cl = lambda v: "" if pd.isna(v) else str(v).strip()
print("Documentos:", len(df))

def parse(text):
    m = re.search(r"Categor[ií]a principal\**\s*:\**\s*\[?\s*(T[1-5])", text, re.I)
    s = re.search(r"Categor[ií]as secundarias[^:]*:\**\s*(.*)", text, re.I)
    sec = ",".join(re.findall(r"T[1-5]", s.group(1))) if s else ""
    return (m.group(1).upper() if m else ""), sec

def classify(i):
    r = df.loc[i]
    user = (f"TÍTULO: {cl(r['Título'])}\nRESUMEN: {cl(r['Resumen (texto literal)'])}\n"
            f"PALABRAS CLAVE: {cl(r['Palabras clave (texto literal)']) or 'No proporcionadas'}")
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model=MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}])
            txt = resp.choices[0].message.content
            main, sec = parse(txt)
            return {"fila": i, "ID original": r.get("ID original", i), "etiqueta": r["Clasificacion tematica"],
                    "deepseek": main, "secundarias": sec, "respuesta": txt,
                    "modelo_api": resp.model, "fecha": datetime.datetime.utcnow().isoformat(timespec="seconds"),
                    "tokens_entrada": resp.usage.prompt_tokens, "tokens_salida": resp.usage.completion_tokens}
        except Exception as e:
            time.sleep(2 ** attempt)
            last = str(e)
    return {"fila": i, "ID original": r.get("ID original", i), "etiqueta": r["Clasificacion tematica"],
            "deepseek": "", "secundarias": "", "respuesta": f"ERROR: {last}", "modelo_api": "", "fecha": "",
            "tokens_entrada": 0, "tokens_salida": 0}

In [ ]:
# Ejecución con reanudación: si se corta, vuelva a ejecutar esta celda y continúa donde quedó
done = pd.read_csv(OUT_CSV) if os.path.exists(OUT_CSV) else pd.DataFrame()
ok = set(done.loc[done["deepseek"].fillna("") != "", "fila"]) if len(done) else set()
todo = [i for i in range(len(df)) if i not in ok]
print(f"Pendientes: {len(todo)}")
rows = done[done["fila"].isin(ok)].to_dict("records") if len(done) else []
with ThreadPoolExecutor(WORKERS) as ex:
    futs = [ex.submit(classify, i) for i in todo]
    for n, f in enumerate(as_completed(futs), 1):
        rows.append(f.result())
        if n % 10 == 0 or n == len(futs):
            pd.DataFrame(rows).sort_values("fila").to_csv(OUT_CSV, index=False)
            print(f"  {n}/{len(futs)} guardados")
res = pd.read_csv(OUT_CSV)
print("Sin etiqueta válida:", (res["deepseek"].fillna("") == "").sum())

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
L = ["T1","T2","T3","T4","T5"]
v = res[res["deepseek"].isin(L)]
print("Modelo según la API:", v["modelo_api"].unique(), "| fechas:", v["fecha"].min(), "a", v["fecha"].max())
print(f"n={len(v)}  exactitud={accuracy_score(v.etiqueta, v.deepseek):.3f}  macro-F1={f1_score(v.etiqueta, v.deepseek, average='macro'):.3f}")
print(classification_report(v.etiqueta, v.deepseek, labels=L, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(v.etiqueta, v.deepseek, labels=L), index=L, columns=L))
print("Tokens totales: entrada", res.tokens_entrada.sum(), "| salida", res.tokens_salida.sum())

In [ ]:
from google.colab import files
files.download(OUT_CSV)   # envíe este archivo para el análisis completo